# Exploratory Notebook

## Basic imports

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from plotting_utils import FlatmapMapper, FigureConfig, FlatmapPlotter

from pathlib import Path

sys.path.append('../')
from data_loading import (
    DataSequence, ContextManager, EmbeddingManager, 
    file_io, config, prepare_data, preprocessing
)

from main import pipeline

root_dir = Path('/mnt/antares_raid/home/simonee/nonsense/')

/Users/simoneesposito/Desktop/Uni/WiSe 25_26 FINAL BOSS/nonsense/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataseqs = file_io.load_dataseqs(root_dir/config.DATASEQ_PATH, config.STORIES)
context_mgr = ContextManager(root_dir/config.CONTEXTS_FILE)
contexts = context_mgr.get_or_create_contexts(
    mode='remove-pos-noun',
    stories=config.STORIES,
    dataseqs=dataseqs,
    overwrite=True
)

In [ ]:
import itertools

def pos_combinations(pos):
    pos_permutation = []

    for r in range(1, len(pos) + 1):
        # itertools.combinations(iterable, length) - no repetition
        for p in itertools.combinations(pos, r):
            pos_permutation.append(list(p))
    
    pos_permutation = ['remove-pos-' + '-'.join(x) for x in pos_permutation]
    return pos_permutation


## Generate Results (All Modes)

In [ ]:
subject = 'subject07'
modality = 'listening'
#modes = ['english1000', 'baseline', 'remove-valleys', 'remove-peaks', 'zero', 'random', 'shuffle']
pos = ['noun', 'verb']
#modes = modes + pos_combinations(pos)
modes = [f'remove-pos-{p}' for p in pos]

In [ ]:
dataseqs = file_io.load_dataseqs(root_dir/config.DATASEQ_PATH, config.STORIES)
mode = 'remove-pos-verb'
context_mgr = ContextManager(root_dir/config.CONTEXTS_FILE)
contexts = context_mgr.get_or_create_contexts(
    mode=mode,
    stories=config.STORIES,
    dataseqs=dataseqs,
    overwrite=True,
)
# Handle embeddings
embedding_mgr = EmbeddingManager(root_dir/config.EMBEDDINGS_FILE)
embeddings = embedding_mgr.get_or_create_embeddings(
    mode=mode,
    stories=config.STORIES,
    dataseqs=dataseqs,
    contexts=contexts,
    overwrite=True,
)

In [5]:
subject = 'subject07'
modality = 'listening'
modes = ['baseline', 'noun', 'verb']
for mode in modes:
    dataseqs = file_io.load_dataseqs(root_dir/config.DATASEQ_PATH, config.STORIES)
    context_mgr = ContextManager(root_dir/'data/features/masks.npz')
    contexts = context_mgr.get_or_create_contexts(
        mode=mode,
        stories=config.STORIES,
        dataseqs=dataseqs,
    )
    embedding_mgr = EmbeddingManager(root_dir/'data/features/embeddings_masked.npz')
    embeddings = embedding_mgr.get_or_create_embeddings(
        mode=mode,
        stories=config.STORIES,
        dataseqs=dataseqs,
        contexts=contexts,
        verbose=False,
    )
"""
pipeline(
    subject=subject,
    modality=modality,
    modes=modes,
    dataseqs=dataseqs,
    contexts=contexts,
    output_dir=root_dir/'outputs'/'result_masks',
    embeddings=embeddings,
    overwrite=True
)
"""

"\npipeline(\n    subject=subject,\n    modality=modality,\n    modes=modes,\n    dataseqs=dataseqs,\n    contexts=contexts,\n    output_dir=root_dir/'outputs'/'result_masks',\n    embeddings=embeddings,\n    overwrite=True\n)\n"

In [14]:
np.load(root_dir/'data/features/embeddings_masked.npz', allow_pickle=True)['noun'].item().keys()

dict_keys(['alternateithicatom_en', 'avatar_en', 'howtodraw_en', 'legacy_en', 'life_en', 'myfirstdaywiththeyankees_en', 'naked_en', 'odetostepfather_en', 'souls_en', 'undertheinfluence_en', 'wheretheressmoke_en'])

In [ ]:
subject = 'subject07'
modality = 'listening'
modes = ['baseline', 'noun', 'verb']
results = {}
for mode in modes:
    results[mode] = pipeline(
        [subject],
        modality,
        mode,
        config.STORIES,
        config.NUIS_LISTENING,
        config.NUIS_READING,
        root_dir,
        output_dir=root_dir / config.OUTPUT_DIR / "results_masks",
        contexts_file = root_dir/'data/features/masks.npz',
        embeddings_file = root_dir/'data/features/embeddings_masked.npz',
    )

Mode 'baseline': 0 missing stories out of 11.
Stacking and delaying features for training... ✓
Stacking and delaying features for validation... ✓
Stacking responses... ✓
Building feature groups... ✓
Results for mode 'baseline' already exist for subject 'subject07'. Skipping...
Averaged correlation (r) across voxels: 0.2189618945121765
Script execution finished.


Mode 'noun': 0 missing stories out of 11.
Stacking and delaying features for training... ✓
Stacking and delaying features for validation... ✓
Stacking responses... ✓
Building feature groups... ✓
Results for mode 'noun' already exist for subject 'subject07'. Skipping...
Averaged correlation (r) across voxels: 0.19899950921535492
Script execution finished.


Mode 'verb': 0 missing stories out of 11.
Stacking and delaying features for training... ✓
Stacking and delaying features for validation... ✓
Stacking responses... ✓
Building feature groups... ✓
Fitting MultipleKernelRidgeCV model...
[........................................] 100% | 248.37 sec | 100 random sampling with cv | 
Model fitting complete.
Processing results... ✓
Computing kernels for validation... ✓
Calculating correlations... ✓


/home/simonee/miniconda3/envs/nonsense/lib/python3.11/site-packages/himalaya/backend/_utils.py:138: UserWarning: GPU backend torch_cuda is much faster with single precision floats (float32), got input in torch.float64. Consider casting your data to float32.
  warnings.warn(


Performing permutation tests...

  8%|▊         | 190/2500 [00:04<00:48, 47.76it/s]

In [ ]:
for modality in ['listening', 'reading']:
    for mode in modes:
        print("="*60)
        print(f"Processing mode: {mode} | Modality: {modality}")
        print("-"*60)
        try:
            pipeline(
                config.SUBJECTS,
                modality,
                mode,
                config.STORIES,
                config.NUIS_LISTENING,
                config.NUIS_READING,
                root_dir,
                output_dir=root_dir / config.OUTPUT_DIR / "results",
                window_size=20,
                override=True
            )
        except Exception as e:
            print(f"Error processing mode {mode}: {e}")
        


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import numpy as np
sys.path.append('../')
from data_loading.embedding_generator import lexical_embeddings
from data_loading import file_io, config, load_features, load_responses, prepare_features, prepare_data
from pathlib import Path
from main import pipeline, perform_group_ridge, get_scores_and_prediction, permutation_test, apply_fdr_correction
from himalaya.scoring import correlation_score_split
from plotting_utils import FlatmapMapper, FigureConfig, FlatmapPlotter, BivariateStrategy

In [ ]:
def mini_pipeline(subject, modality, mode, embeddings, root_dir):
    stories_train = config.STORIES[:-1]
    stories_val = config.STORIES[-1:]
    F_trn = load_features("trn", root_dir/ config.FEATURE_PATH, stories=stories_train)
    F_val = load_features("val", root_dir/ config.FEATURE_PATH, stories=stories_val)
    R_trn = load_responses("trn",root_dir/ config.RESPONSE_PATH, [subject], modality, stories=stories_train)
    R_val = load_responses("val",root_dir/ config.RESPONSE_PATH, [subject], modality, stories=stories_val)

    F_trn = prepare_features(F_trn, embeddings, mode, stories_train)
    F_val = prepare_features(F_val, embeddings, mode, stories_val)

    use_keys = [mode] + (config.NUIS_LISTENING if modality == "listening" else config.NUIS_READING)
    X_trn, Y_trn, X_val, Y_val, groups, story_ids = prepare_data(
        R_trn,
        R_val,
        F_trn,
        F_val,
        stories_train,
        stories_val,
        use_keys,
        delays=np.arange(0, 9, 2),
    )

    results_grr = perform_group_ridge(
        X_trn,
        Y_trn[subject],
        groups,
        story_ids[subject],
        alphas=np.logspace(-10, 10, 21),
        use_keys=use_keys,
    )
    r, r2, predictions = get_scores_and_prediction(results_grr, X_val, X_trn, Y_val[subject], groups, use_keys)
    score_names = ["r"]
    score_funcs = [correlation_score_split]
    pvalues = {}
    for score_name, score_func in zip(score_names, score_funcs):
        pvalues[score_name] = permutation_test(Y_val[subject], predictions, score_func, 2500)
    print(" ✓")
    fdr = apply_fdr_correction(pvalues)
    results = {
        "r": r,
        "r2": r2,
        "fdr": fdr,
    }
    averaged_r = np.nanmean(results["r"][results["fdr"]["r"]["included_voxels_indices"]], axis=0)
    print(f"Averaged r across significant voxels: {averaged_r:.4f}")
    return results

def get_significant_voxels(results, mode):
    sig = np.array(results[mode]['r'])
    sig[results[mode]['fdr']['r']['excluded_voxels_indices']] = np.nan
    return sig

In [ ]:
root_dir = Path('../')
subject = 'subject07'
modality = 'listening'
dataseqs = file_io.load_dataseqs(root_dir/config.DATASEQ_PATH, config.STORIES)

baseline_embeddings = lexical_embeddings(
    dataseqs=dataseqs,
    embedding_path=root_dir / config.FEATURE_PATH/ 'english1000.npz',
    pos_tag=None,
    downsamp=True
)
no_nouns_embeddings = lexical_embeddings(
    dataseqs=dataseqs,
    embedding_path=root_dir / config.FEATURE_PATH/ 'english1000.npz',
    pos_tag=['NOUN', 'PROPN'],
    downsamp=True
)
no_verbs_embeddings = lexical_embeddings(
    dataseqs=dataseqs,
    embedding_path=root_dir / config.FEATURE_PATH/ 'english1000.npz',
    pos_tag=['VERB', 'AUX'],
    downsamp=True
)

In [ ]:
embeddings = {
    'baseline': baseline_embeddings,
    'remove-pos-noun': no_nouns_embeddings,
    'remove-pos-verb': no_verbs_embeddings
}
for modality in ['reading', 'listening']
    for subject in config.SUBJECTS:
        print(f"Processing subject: {subject}")
        results = {}
        for mode in ['baseline', 'remove-pos-noun', 'remove-pos-verb']:
            print(f"  Mode: {mode}")
            results[mode] = mini_pipeline(subject, modality, mode, embeddings[mode], root_dir)
            sig_voxels = get_significant_voxels(results, mode)
            print(f"    Significant voxels (non-NaN): {np.sum(~np.isnan(sig_voxels))}")
        file_io.save_results(
            root_dir/config.OUTPUT_DIR / "results_control",
            modality, 
            subject,
            results,
        )

In [ ]:
results_baseline = file_io.load_results(root_dir / config.OUTPUT_DIR / "results_test", modality, subject)['baseline']
results_no_nouns = file_io.load_results(root_dir / config.OUTPUT_DIR / "results_test", modality, subject)['remove-pos-noun']
results_no_verbs = mini_pipeline(subject, modality, 'remove-pos-verb', no_verbs_embeddings, root_dir)

In [ ]:
from plotting_utils import SingleCorrelationStrategy, FlatmapMapper, FigureConfig, FlatmapPlotter
mapper = FlatmapMapper(subject, root_dir/config.MAPPER_PATH)
fig_config = FigureConfig(
    figsize=(12, 8), 
    save=False, 
    smooth=True, 
    output_dir=root_dir/config.OUTPUT_DIR/"images"/"same_cmap"
)
plotter = FlatmapPlotter(mapper, fig_config)
_= plotter.plot(
    data = [get_significant_voxels(results, 'baseline')],
    strategy = SingleCorrelationStrategy(vmax=0.5),
    title = f"Correlation for baseline",
)
_= plotter.plot(
    data = [get_significant_voxels(results, 'remove-pos-noun')],
    strategy = SingleCorrelationStrategy(vmax=0.5),
    title = f"Correlation for no-nouns",
)
_= plotter.plot(
    data = [get_significant_voxels(results, 'remove-pos-verb')],
    strategy = SingleCorrelationStrategy(vmax=0.5),
    title = f"Correlation for no-verbs",
)

In [ ]:
for subject in config.SUBJECTS:
    if subject == 'subject04':
        continue
    results = file_io.load_results(root_dir / config.OUTPUT_DIR / "results_control", modality, subject)
    baseline = get_significant_voxels(results, 'baseline')
    nouns = baseline - get_significant_voxels(results, 'remove-pos-pct-noun')
    verbs = baseline - get_significant_voxels(results, 'remove-pos-pct-verb')

    fig_config = FigureConfig(
        figsize=(12, 8),
        smooth=False,
        show=True,
        show_regions=True,
        save= False,
        output_dir=root_dir / config.IMAGES_DIR/ modality / "2D Plots"
    )
    mapper = FlatmapMapper(subject, root_dir/config.MAPPER_PATH)
    plotter = FlatmapPlotter(mapper, fig_config)
    _ = plotter.plot(
        [nouns, verbs],
        strategy=BivariateStrategy(vmax=0.1, labels=['nouns', 'verbs']),
        title = f'Nouns vs Verbs {subject}',
        output_filename = f'{subject}.png'
    )

In [ ]:
for subject in config.SUBJECTS:
    if subject == 'subject04':
        continue
    results = file_io.load_results(root_dir / config.OUTPUT_DIR / "results_control", modality, subject)
    baseline = results['baseline']['r']
    nouns = baseline - results['remove-pos-pct-noun']['r']
    verbs = baseline - results['remove-pos-pct-verb']['r']

    fig_config = FigureConfig(
        figsize=(12, 8),
        smooth=False,
        show=True,
        show_regions=True,
        save= False,
        output_dir=root_dir / config.IMAGES_DIR/ modality / "2D Plots"
    )
    mapper = FlatmapMapper(subject, root_dir/config.MAPPER_PATH)
    plotter = FlatmapPlotter(mapper, fig_config)
    _ = plotter.plot(
        [nouns, verbs],
        strategy=BivariateStrategy(vmax=0.1, labels=['nouns', 'verbs']),
        title = f'Nouns vs Verbs {subject}',
        output_filename = f'{subject}.png'
    )

## Plot Results Univariate

In [ ]:
subject = 'subject07'
modality = 'listening'
mode = 'baseline'

In [ ]:
from plotting_utils import SingleCorrelationStrategy
mapper = FlatmapMapper(subject, root_dir/config.MAPPER_PATH)
fig_config = FigureConfig(
    figsize=(12, 8), 
    save=True, 
    smooth=True, 
    output_dir=root_dir/config.OUTPUT_DIR/"images"/"same_cmap"
)
plotter = FlatmapPlotter(mapper, fig_config)

In [ ]:
plotter.plot(
    data = [results[mode]['r']],
    strategy = SingleCorrelationStrategy(vmax=0.5),
    title = f"Correlation for {mode}",
)

In [ ]:
for mode in modes:
    pu.plot_correlation_on_flatmap(
        subject,
        modality,
        mode,
        results[mode],
        root_dir/config.MAPPER_PATH,
        root_dir/config.OUTPUT_DIR/"images"/"same_cmap",
        show = False
    )

## Plot Results Bivariate

In [ ]:
subject = 'subject07'
modality = 'reading'
modes = ['remove-valleys', 'remove-peaks', 'zero', 'random', 'shuffle', 
         'remove-pos-noun', 'remove-pos-verb', 'remove-pos-adj', 'remove-pos-adv']

In [ ]:
results = file_io.load_results(
    root_dir / config.OUTPUT_DIR / "results",
    modality,
    subject
)

In [ ]:
correlation_baseline = results['baseline']['r']

In [ ]:
for mode in modes:
    correlation_comparison = results[mode]['r']
    pu.plot_bivariate_flatmap(
        subject,
        modality,
        correlation_baseline,
        correlation_comparison,
        "baseline",
        mode,
        root_dir/config.MAPPER_PATH,
        root_dir/config.IMAGES_DIR/'bivariate',
        show = True
    )

## Plot Results Colored

In [ ]:
subject = 'subject07'
modality = 'listening'
modes = ['remove-pos-noun', 'remove-pos-verb', 'remove-pos-adj', 'remove-pos-adv', 'remove-pos-pron', 'remove-pos-intj']
control = ['remove-pos-pct-noun', 'remove-pos-pct-verb', 'remove-pos-pct-adj', 'remove-pos-pct-adv', 'remove-pos-pct-pron', 'remove-pos-pct-intj']
names = ['no-noun', 'no-verb', 'no-adj', 'no-adv', 'no-pron', 'no-intj']

In [ ]:
mapper = FlatmapMapper(subject, root_dir/config.MAPPER_PATH)
fig_config = FigureConfig(
    figsize=(12, 8), 
    save=True, 
    smooth=True, 
    output_dir=root_dir/config.OUTPUT_DIR/"images"/"same_cmap"
)
plotter = FlatmapPlotter(mapper, fig_config)   

In [ ]:
results = file_io.load_results(
    root_dir / config.OUTPUT_DIR / "results",
    modality,
    subject
)

NameError: name 'file_io' is not defined

In [ ]:
for modality in ['listening', 'reading']:
    for subject in config.SUBJECTS:
        try:
            results = file_io.load_results(
                root_dir / config.OUTPUT_DIR / "results",
                modality,
                subject
            )
        except Exception as e:
            print(f"Error loading results for {modality} - {subject}: {e}")
            continue
        
        modes = results.keys()
        for mode in modes:
            results[mode] = {
                'r': results[mode]['r'],
                'r2': results[mode]['r2'],
                'fdr': results[mode]['fdr']
            }
        file_io.save_results(
            root_dir / config.OUTPUT_DIR / "results",
            modality,
            subject,
            results
        )

In [ ]:
e = 0.5
correlations = [(results[mode]['r']-results['baseline']['r'])/(results['baseline']['r']+e) for mode in modes]
correlations_sig = []
for i in range(len(modes)):
    temp = correlations[i].copy()
    temp[results[modes[i]]['fdr']['r']['excluded_voxels_indices']] = 0
    correlations_sig.append(temp)
    
control_correlations = [(results[mode]['r']-results['baseline']['r'])/(results['baseline']['r']+e) for mode in control]
control_correlations_sig = []   
for i in range(len(control)):
    temp = control_correlations[i].copy()
    temp[results[control[i]]['fdr']['r']['excluded_voxels_indices']] = 0
    control_correlations_sig.append(temp)


In [ ]:
from plotting_utils import LoserTakesItAllStrategy

_ = plotter.plot(
    data = correlations_sig,
    strategy = LoserTakesItAllStrategy(names=names),
    title = f"Correlation for {mode}",
    output_filename = 'ltia actual'
)
_ = plotter.plot(
    data = control_correlations_sig,
    strategy = LoserTakesItAllStrategy(names=names),
    title = f"Control Correlation for {mode}",
    output_filename = 'ltia control'
)

In [ ]:
pu.plot_loser_takes_it_all(
    subject,
    modality,
    control_correlations_sig[:],
    names[:],
    root_dir/config.MAPPER_PATH,
    root_dir/config.IMAGES_DIR/'bivariate',
    smooth = True,
    regions = False,
    show = True,
)

In [ ]:
pu.plot_loser_takes_it_all_margin(
    subject,
    modality,
    control_correlations_sig[:],
    names[:],
    root_dir/config.MAPPER_PATH,
    root_dir/config.IMAGES_DIR/'POS',
    regions = False,
    show = True,
)

In [ ]:
pu.plot_bivariate_flatmap(
    subject,
    modality,
    correlations[0],
    correlations[1],
    names[0],
    names[1],
    root_dir/config.MAPPER_PATH,
    root_dir/config.IMAGES_DIR/'bivariate',
    show = True
)

In [ ]:
plot_margin_comparison(correlations_sig[0], control_correlations_sig[0])

In [ ]:
def compute_margin(corr):
    stacked = np.stack(corr, axis=0)
    worst_indices = np.argsort(stacked, axis=0)[0]
    second_worst_indices = np.argsort(stacked, axis=0)[1]

    worst_vals = stacked[worst_indices, np.arange(stacked.shape[1])]
    second_worst_vals = stacked[second_worst_indices, np.arange(stacked.shape[1])]

    margin = second_worst_vals- worst_vals
    return margin

In [ ]:
def plot_margin_comparison(margins_actual, margins_control, title="Margin Distribution: Actual vs Control"):
    # 1. Flatten and Clean Data (Remove NaNs and Zeros)
    actual_clean = margins_actual.flatten()[~np.isnan(margins_actual.flatten()) & (margins_actual.flatten() != 0)]
    control_clean = margins_control.flatten()[~np.isnan(margins_control.flatten()) & (margins_control.flatten() != 0)]

    # 2. Create a Long-form DataFrame (Required for Seaborn)
    df_actual = pd.DataFrame({'Margin': actual_clean, 'Condition': 'Actual'})
    df_control = pd.DataFrame({'Margin': control_clean, 'Condition': 'Control'})
    df = pd.concat([df_actual, df_control])

    # 3. Plotting
    plt.figure(figsize=(8, 6))
    
    # We use a boxplot to show the quartiles
    # fliersize=1 makes the outlier dots small so they don't overwhelm the plot
    ax = sns.boxplot(x='Condition', y='Margin', data=df, 
                     palette={'Actual': '#E6194B', 'Control': '#A9A9A9'},
                     fliersize=1.5, width=0.5)

    # Optional: Add a 'notch' to show the confidence interval around the median
    # sns.boxplot(..., notch=True)

    plt.title(title, fontsize=14, pad=15)
    plt.ylabel("Margin Magnitude (Specificity)", fontsize=12)
    plt.xlabel("Experiment Type", fontsize=12)
    plt.grid()
    sns.despine() # Makes the plot look cleaner for a thesis
    plt.tight_layout()
    plt.show()

In [ ]:
margins_actual.shape

In [ ]:
margins_actual = compute_margin(correlations_sig)
margins_control = compute_margin(control_correlations_sig)
plot_margin_comparison(margins_actual, margins_control)

In [ ]:
diff = margins_actual - margins_control
print(f"Margin Difference: Mean={np.nanmean(diff):.4f}, Median={np.nanmedian(diff):.4f}, Std={np.nanstd(diff):.4f}")
v_max = np.nanpercentile(np.abs(diff), 95)
normalized = diff / (v_max + 1e-10)
diff_normal = np.clip(normalized, -1, 1)

In [ ]:
pu.plot_correlation_on_flatmap(
    subject,
    modality,
    "drop_control",
    contro,
    root_dir/config.MAPPER_PATH,
    root_dir/config.OUTPUT_DIR/"images"/"margin_diff",
    show = True,
    regions=True
)

In [ ]:
pu.plot_correlation_on_flatmap(
    subject,
    modality,
    "diff_normal",
    diff_normal,
    root_dir/config.MAPPER_PATH,
    root_dir/config.OUTPUT_DIR/"images"/"margin_diff",
    show = True,
    regions=True
)

## Story Analysis

In [ ]:
import spacy
import matplotlib.pyplot as plt
from collections import defaultdict

def analyze_pos(text, pos_tags):
    """Analyze parts of speech in text and return percentages."""
    doc = nlp(text)
    pos_counts = defaultdict(int)
    
    for token in doc:
        if token.pos_ in pos_tags:
            pos_counts[token.pos_] += 1
    
    total = sum(pos_counts.values())
    
    percentages = {tag: (pos_counts[tag] / total * 100) if total > 0 else 0 for tag in pos_tags}
    
    return percentages


In [ ]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Sample stories (replace with your actual data)
stories = file_io.load_dataseqs(root_dir / config.DATASEQ_PATH, config.STORIES)
stories = {'story ' + str(i): ' '.join(v.data) for i, v in enumerate(stories.values())}

In [ ]:
# Analyze each story
pos = {}
pos_tags = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'INTJ', 'PROPN', 'ADP', 'CONJ', 'DET', 'NUM', 'PART', 'SCONJ', 'PUNCT', 'SYM', 'X']
for story_name, text in stories.items():
    pos[story_name] = analyze_pos(text, pos_tags)

# Prepare data for plotting
story_names = list(pos.keys())
nouns = [pos[s]["NOUN"] for s in story_names]
verbs = [pos[s]["VERB"] for s in story_names]
adjectives = [pos[s]["ADJ"] for s in story_names]
adverbs = [pos[s]["ADV"] for s in story_names]
pronouns = [pos[s]["PRON"] for s in story_names]
interjections = [pos[s]["INTJ"] for s in story_names]
proper_nouns = [pos[s]["PROPN"] for s in story_names]
adpositions = [pos[s]["ADP"] for s in story_names]
conjunctions = [pos[s]["CONJ"] for s in story_names]
determiners = [pos[s]["DET"] for s in story_names]
numerals = [pos[s]["NUM"] for s in story_names]
particles = [pos[s]["PART"] for s in story_names]
subordinating_conjunctions = [pos[s]["SCONJ"] for s in story_names]
punctuations = [pos[s]["PUNCT"] for s in story_names]
symbols = [pos[s]["SYM"] for s in story_names]
others = [pos[s]["X"] for s in story_names]

# Create bar chart
x = range(len(story_names))
width = 0.1

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar([i - 2.5*width for i in x], nouns, width, label="Nouns")
ax.bar([i - 1.5*width for i in x], verbs, width, label="Verbs")
ax.bar([i - 0.5*width for i in x], adjectives, width, label="Adjectives")
ax.bar([i + 0.5*width for i in x], adverbs, width, label="Adverbs")
ax.bar([i + 1.5*width for i in x], pronouns, width, label="Pronouns")
ax.set_xlabel("Stories")
ax.set_ylabel("Percentage (%)")
ax.set_title("Parts of Speech Distribution by Story")
ax.set_xticks(x)
ax.set_xticklabels(story_names)
ax.legend()
ax.set_ylim(0, 50)

plt.tight_layout()
plt.show()


In [ ]:
import warnings
import numpy as np

def get_loser_counts(correlations, names):
    stacked = np.stack(correlations, axis=0)

    # Identify all-NaN columns before calling nanargmin
    all_nan_mask = np.all(np.isnan(stacked), axis=0)

    # Replace all-NaN columns with a safe value (e.g., infinity)
    stacked_safe = stacked.copy()
    stacked_safe[:, all_nan_mask] = np.inf

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning)
        loser_indices = np.nanargmin(stacked_safe, axis=0)

    # Mark all-NaN positions with -1 (or np.nan)
    loser_indices[all_nan_mask] = -1

    # Count occurrences for each condition
    counts = np.bincount(loser_indices[loser_indices >= 0].astype(int), minlength=len(names))

    # turn counts into percentages
    counts = counts / np.sum(counts) * 100
    return counts

In [ ]:
# Create bar chart with color coding
counts = get_loser_counts(correlations_sig, names)
x = np.arange(len(names))
width = 0.6

colors = ["#E6194B", "#3CB44B", "#FFE119", "#4363D8", "#F58231", "#911EB4"] 

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x, counts, width=width, color=colors, edgecolor='black', alpha=0.8)

ax.set_xlabel('Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Minimum Correlations', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare POS frequency data for comparison
pos = {}
pos_labels = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'INTJ']
for story_name, text in stories.items():
    pos[story_name] = analyze_pos(text, pos_labels)
pos_mean = np.array([list(v.values()) for v in pos.values()]).mean(axis=0)

# Prepare control correlations for comparison
control_counts = get_loser_counts(control_correlations_sig, names)

# Prepare data for comparison
pos_mean_values = pos_mean 

# Create side-by-side bar plot
x = np.arange(len(names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))

# Plot POS frequency values
ax.bar(x - width, pos_mean_values, width, label='Mean POS Distribution', 
       color='coral', edgecolor='black', alpha=0.8)

# Plot control counts (from loser_indices analysis)
ax.bar(x, control_counts, width, label='Loser Takes It All Distribution', 
       color='', edgecolor='black', alpha=0.8)

# Plot actual counts (from loser_indices analysis)
ax.bar(x + width, counts, width, label='Loser Takes It All Distribution', 
       color='steelblue', edgecolor='black', alpha=0.8)


ax.set_xlabel('Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('Comparison: Minimum Correlations vs Mean POS Distribution', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()